# IMPORT LIBRARIES

In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch import optim
from torchvision import models
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

train_df = pd.read_csv("CSVs\\train_images.csv")
test_df = pd.read_csv("CSVs\\test_images.csv")

# TRANSFORM IMAGES

## CONFIG

In [2]:
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0

emotion_to_idx = {
    "Anger": 0,
    "Disgust": 1,
    "Fear": 2,
    "Happy": 3,
    "Neutral": 4,
    "Sad": 5
}

## TRANSFORM

In [3]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    
    transforms.RandomAffine(
        degrees=0,
        translate=(0.03,0.03),
        scale=(0.95,1.05)
    ),

    transforms.ToTensor(),

    transforms.RandomErasing(
        p=0.35,
        scale=(0.02,0.10),
        ratio=(0.3,3.3)
    ),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])


val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [4]:
class ImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = row["path"]
        label = emotion_to_idx[row["emotion"]]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

# LOAD DATA

In [5]:
train_dataset = ImageDataset(train_df, transform=train_transform)

test_dataset = ImageDataset(test_df, transform=val_transform)

In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [7]:
print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

x, y = next(iter(train_loader))
print("Batch image shape:", x.shape)   # [B,3,224,224]
print("Batch label shape:", y.shape)   # [B]

Train samples: 11780
Test samples: 1552
Batch image shape: torch.Size([32, 3, 224, 224])
Batch label shape: torch.Size([32])


# CNN MODEL

In [8]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

Using device: cuda


## RESNET

### MODEL

In [9]:
def resnet50(num_classes=6):
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

    for param in model.parameters():
        param.requires_grad = True

    for layer in [model.layer2, model.layer3, model.layer4]:
        for param in layer.parameters():
            param.requires_grad = True

    in_features = model.fc.in_features

    model.fc = nn.Sequential(  # type: ignore
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, num_classes),
    )

    return model

model = resnet50().to(device)


### CONFIG

In [10]:
WEIGHT_DECAY = 1e-4
LEARNING_RATE = 5e-5
PATIENCE = 5

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.Adam(
    filter(
        lambda p: p.requires_grad,
        model.parameters(),
    ),
    weight_decay=WEIGHT_DECAY,
    lr=LEARNING_RATE,
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=PATIENCE
)

### EVAL FUNC

In [ ]:
def evaluate(loader):
    model.eval()

    y_true = []
    y_pred = []

    total_loss = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)
            total_loss += loss.item()

            preds = torch.argmax(
                outputs,
                dim=1
            )

            y_true.extend(
                labels.cpu().numpy()
            )

            y_pred.extend(
                preds.cpu().numpy()
            )

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(y_true, y_pred)

    f1 = f1_score(
        y_true,
        y_pred,
        average="weighted"
    )

    prec = precision_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    rec = recall_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    return avg_loss, acc, f1, prec, rec

### TRAIN

In [12]:
EPOCHS = 40
PATIENCE = 5

best_val_loss = float("inf")
best_f1 = 0
no_improve = 0

for epoch in range(EPOCHS):

    model.train()
    train_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    val_loss, val_acc, val_f1, val_prec, val_rec = evaluate(
        test_loader
    )

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"LR {current_lr:.6f} | "
        f"Train Loss {train_loss:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Acc {val_acc:.4f} | "
        f"F1 {val_f1:.4f} | "
        f"Prec {val_prec:.4f} | "
        f"Rec {val_rec:.4f}"
    )

    # Save best
    if val_f1 > best_f1:
        best_f1 = val_f1
        no_improve = 0

        torch.save(
            model.state_dict(),
            "best_resnet34_cremad.pth"
        )

    else:
        no_improve += 1

        if no_improve >= PATIENCE:
            print("Early stopping triggered.")
            break

Epoch 1/40 | LR 0.000050 | Train Loss 1.5281 | Val Loss 1.3886 | Acc 0.5161 | F1 0.5012 | Prec 0.5236 | Rec 0.5161
Epoch 2/40 | LR 0.000050 | Train Loss 1.2612 | Val Loss 1.2651 | Acc 0.5780 | F1 0.5759 | Prec 0.5794 | Rec 0.5780
Epoch 3/40 | LR 0.000050 | Train Loss 1.0955 | Val Loss 1.2357 | Acc 0.6082 | F1 0.6058 | Prec 0.6076 | Rec 0.6082
Epoch 4/40 | LR 0.000050 | Train Loss 0.9671 | Val Loss 1.2893 | Acc 0.5986 | F1 0.5931 | Prec 0.6103 | Rec 0.5986
Epoch 5/40 | LR 0.000050 | Train Loss 0.8474 | Val Loss 1.2827 | Acc 0.6102 | F1 0.6099 | Prec 0.6172 | Rec 0.6102
Epoch 6/40 | LR 0.000050 | Train Loss 0.7578 | Val Loss 1.3090 | Acc 0.6160 | F1 0.6128 | Prec 0.6198 | Rec 0.6160
Epoch 7/40 | LR 0.000050 | Train Loss 0.6833 | Val Loss 1.2939 | Acc 0.6186 | F1 0.6170 | Prec 0.6205 | Rec 0.6186
Epoch 8/40 | LR 0.000050 | Train Loss 0.6287 | Val Loss 1.3383 | Acc 0.6095 | F1 0.6053 | Prec 0.6129 | Rec 0.6095
Epoch 9/40 | LR 0.000025 | Train Loss 0.5892 | Val Loss 1.3456 | Acc 0.6179 | F1